1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 답변 생성이 오래걸림
3. 임베딩 --> 벡터 데이터 베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

python.langchain 들어가면 docs 변화 방법 있음

# 1. 문서의 내용을 읽는다

In [ ]:
%pip install python-docx

In [ ]:
from docx import Document

document= Document('./tax.docx')
print(f'document = {dir(Document)}')
full_text = ''
for index, paragraphs in enumerate(document.paragraphs):
    print(f'paragraph == {paragraphs.text}')
    full_text += f'{paragraphs.text}\n'
    

# 2.문서를 쪼갠다

In [ ]:
# %pip install tiktoken
%pip install tokenizers==0.20.0

In [ ]:
from tokenizers import Tokenizer

def split_text(full_text, chunk_size):
    encoder = Tokenizer.from_pretrained("upstage/solar-pro2-tokenizer")
    
    encoding = encoder.encode(full_text)
    total_tokens = encoding.ids
    total_token_count = len(total_tokens)
    
    text_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk = total_tokens[i: i+chunk_size]
        decoded = encoder.decode(chunk)
        text_list.append(decoded)
    
    return text_list

In [ ]:
chunk_list = split_text(full_text, 1500)

In [ ]:
chunk_list

# 3. 임베딩

In [ ]:
%pip install chromadb

In [ ]:
import chromadb

chroma_client = chromadb.Client()

In [ ]:
collection_name = 'tax_collection'

tax_collection = chroma_client.create_collection(collection_name)

In [ ]:
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_embedding = OpenAIEmbeddingFunction(model_name='text-embedding-3-large')

In [ ]:
tax_collection = chroma_client.get_or_create_collection(collection_name, embedding_function= openai_embedding)

In [ ]:
id_list = []
for index in range(len(chunk_list)):
    id_list.append(f'{index}')

In [ ]:
len(id_list)

In [ ]:
len(chunk_list)

In [ ]:
tax_collection.add(documents=chunk_list, ids=id_list)

# 4.유사도 검색

In [ ]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_doc = tax_collection.query(query_texts=query, n_results=3)

In [ ]:
retrieved_doc['documents'][0]

# 5.LLM질의

In [ ]:
%pip install openai

In [85]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"
)
stream = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "system",
            "content": "당신은 한국의 소득세 전문가 입니다. 아래 내용을 참고해서 사용자의 질문에 답변해주세요{retrieved_doc['documents'][0]}"
        }
        ,{
            "role": "user",
            "content": query
        }
    ],
    stream=True,
)

In [87]:
stream